AIM : To implement classification and regression to compare the evaluation metrics by chanaging various parameters.

In [13]:
#CLASSIFICATION USING WINE DATASET(Change in  number of hidden layers and neurons)

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

data = load_wine()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# -------------------------------
# 3. Feature Scaling
# -------------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -------------------------------
# 4. Model Configurations
# -------------------------------
architectures = {
    "3 Hidden Layers": (64, 32, 16),
    "5 Hidden Layers": (128, 64, 32, 16, 8),
    "8 Hidden Layers": (256, 128, 64, 32, 16, 8, 4, 2),
    "10 Hidden Layers": (256, 128, 64, 32, 16, 8, 4, 4, 2, 2),
    "12 Hidden Layers": (256, 128, 64, 32, 16, 8, 8, 4, 4, 2, 2, 2)
}

# -------------------------------
# 5. Fixed Epochs
# -------------------------------
EPOCHS = 1000

# 6. Store Results
results = []

for name, layers in architectures.items():
    model = MLPClassifier(
        hidden_layer_sizes=layers,
        activation="relu",
        max_iter=EPOCHS,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Architecture": layers,
        "No. of Hidden Layers": len(layers),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro"),
        "Recall": recall_score(y_test, y_pred, average="macro"),
        "F1-Score": f1_score(y_test, y_pred, average="macro")
    })

df_results = pd.DataFrame(results)
print("\nPERFORMANCE COMPARISON TABLE\n")
print(df_results.round(4))


/home/student/0040/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/student/0040/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



PERFORMANCE COMPARISON TABLE

                                  Architecture  No. of Hidden Layers  \
0                                 (64, 32, 16)                     3   
1                         (128, 64, 32, 16, 8)                     5   
2              (256, 128, 64, 32, 16, 8, 4, 2)                     8   
3        (256, 128, 64, 32, 16, 8, 4, 4, 2, 2)                    10   
4  (256, 128, 64, 32, 16, 8, 8, 4, 4, 2, 2, 2)                    12   

   Accuracy  Precision  Recall  F1-Score  
0    1.0000     1.0000  1.0000    1.0000  
1    0.9259     0.9393  0.9048    0.9126  
2    0.9815     0.9848  0.9825    0.9832  
3    0.7222     0.4979  0.6491    0.5582  
4    0.7407     0.5225  0.6667    0.5793  


/home/student/0040/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/student/0040/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [14]:
# CLASSIFICATION USING WINE DATASET(SOFTMAX FOR OUTPUT LAYER)

import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU
from tensorflow.keras.utils import to_categorical

data = load_wine()
X, y = data.data, data.target

y = to_categorical(y)  # One-hot encoding

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def build_classification_model(activation):
    model = Sequential()

    if activation == "leaky_relu":
        model.add(Dense(64, input_shape=(X_train.shape[1],)))
        model.add(LeakyReLU(alpha=0.01))
        model.add(Dense(32))
        model.add(LeakyReLU(alpha=0.01))
        model.add(Dense(16))
        model.add(LeakyReLU(alpha=0.01))
    else:
        model.add(Dense(64, activation=activation, input_shape=(X_train.shape[1],)))
        model.add(Dense(32, activation=activation))
        model.add(Dense(16, activation=activation))

    # Output Layer (Softmax)
    model.add(Dense(3, activation="softmax"))

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

activations = ["sigmoid", "tanh", "relu", "leaky_relu"]
results_cls = []

for act in activations:
    model = build_classification_model(act)
    model.fit(X_train, y_train, epochs=100, batch_size=16, verbose=0)

    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = np.argmax(y_test, axis=1)

    results_cls.append({
        "Activation": act,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro"),
        "Recall": recall_score(y_true, y_pred, average="macro"),
        "F1-Score": f1_score(y_true, y_pred, average="macro")
    })

df_cls = pd.DataFrame(results_cls)
print("\nCLASSIFICATION RESULTS (SOFTMAX OUTPUT)\n")
print(df_cls.round(4))


/home/student/0040/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


/home/student/0040/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


/home/student/0040/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step


/home/student/0040/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/student/0040/lib/python3.12/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step

CLASSIFICATION RESULTS (SOFTMAX OUTPUT)

   Activation  Accuracy  Precision  Recall  F1-Score
0     sigmoid    0.9815     0.9833  0.9841    0.9833
1        tanh    0.9815     0.9833  0.9841    0.9833
2        relu    1.0000     1.0000  1.0000    1.0000
3  leaky_relu    1.0000     1.0000  1.0000    1.0000


In [12]:
# REGRESSION USING CALIFORNIA HOUSING DATASET(LINEAR FUNCTION FOR OUTPUT LAYER)

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU

data = fetch_california_housing()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def build_regression_model(activation):
    model = Sequential()

    if activation == "leaky_relu":
        model.add(Dense(64, input_shape=(X_train.shape[1],)))
        model.add(LeakyReLU(alpha=0.01))
        model.add(Dense(32))
        model.add(LeakyReLU(alpha=0.01))
        model.add(Dense(16))
        model.add(LeakyReLU(alpha=0.01))
    else:
        model.add(Dense(64, activation=activation, input_shape=(X_train.shape[1],)))
        model.add(Dense(32, activation=activation))
        model.add(Dense(16, activation=activation))

    # Output Layer (Linear)
    model.add(Dense(1, activation="linear"))

    model.compile(
        optimizer="adam",
        loss="mse"
    )
    return model

results_reg = []

for act in activations:
    model = build_regression_model(act)
    model.fit(X_train, y_train, epochs=100, batch_size=16, verbose=0)

    y_pred = model.predict(X_test).flatten()

    results_reg.append({
        "Activation": act,
        "MSE": mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2 Score": r2_score(y_test, y_pred)
    })

df_reg = pd.DataFrame(results_reg)
print("\nREGRESSION RESULTS (LINEAR OUTPUT)\n")
print(df_reg.round(4))


/home/student/0040/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 730us/step


/home/student/0040/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 819us/step


/home/student/0040/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 791us/step


/home/student/0040/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/student/0040/lib/python3.12/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


194/194 ━━━━━━━━━━━━━━━━━━━━ 0s 771us/step

REGRESSION RESULTS (LINEAR OUTPUT)

   Activation     MSE     MAE  R2 Score
0     sigmoid  0.2946  0.3809    0.7755
1        tanh  0.2592  0.3535    0.8025
2        relu  0.2648  0.3533    0.7983
3  leaky_relu  0.2532  0.3325    0.8071


RESULT : Thus, implementation of classification and regression to compare the evaluation metrics by chanaging various parameters.